# Day 4 — Unity Catalog: Security, Grants & Lineage

Allow about 2–3 hours. Later sections (metadata probes, reading topics, quiz) can be shortened or done outside class if time is limited.

This notebook covers RBAC layers, the usual grant chain for views, `SHOW GRANTS`, example `GRANT`/`REVOKE` text, `information_schema` where available, lineage in the catalog UI and optional system views, audit concepts, and masked/projection patterns from notebook 01.

Prerequisites: complete notebook 01 through the views in section 3 (all view types created).

External training materials on RBAC, lineage, and workspace permissions are often aligned with this content; paths here use ABFS and UC views as in the rest of this course.

## Setup — reconnect + paths

In [ ]:
%run ./02-Mount-Azure-Data-Lake

In [ ]:
BASE_PATH   = base_path
CATALOG     = "main"
STUDENT_ID  = "u01"
SCHEMA_NAME = f"day04_{STUDENT_ID}_lab"
FULL_SCHEMA = f"{CATALOG}.{SCHEMA_NAME}"
DAY03_SILVER = BASE_PATH + "/day03/silver/flights_clean"
DAY03_GOLD   = BASE_PATH + "/day03/gold/flights_by_destination"

SILVER_V    = f"{FULL_SCHEMA}.flights_silver_v"
GOLD_V      = f"{FULL_SCHEMA}.flights_gold_v"
DEST_V      = f"{FULL_SCHEMA}.flights_silver_dest_only_v"
US_V        = f"{FULL_SCHEMA}.flights_silver_us_only_v"
TAGGED_V    = f"{FULL_SCHEMA}.flights_silver_tagged_v"
MASKED_V    = f"{FULL_SCHEMA}.flights_silver_masked_v"

print("Schema:", FULL_SCHEMA)
print("Views :", SILVER_V, GOLD_V, DEST_V, US_V, TAGGED_V, MASKED_V)

## 1 — RBAC layers & privilege model (M-7 Lab 19 / M-1 Lab 02)

### 1a — Four layers of access control in Databricks

| Layer | Scope | Managed by | This course |
|-------|-------|-----------|-------------|
| **Account / identity** | Users, groups, SPs (Entra ID) | Account admin | Set up by admins |
| **Workspace** | Notebooks, dashboards, folders | Workspace admin | Your folder `day04-uXX` |
| **Compute** | Cluster policies, warehouse access | Admin / cluster creator | UC-compatible cluster |
| **Data (UC)** | Catalogs, schemas, tables, views, volumes, ext. locations | Object owner / metastore admin | **Main focus** |

**Principle of Least Privilege (PoLP):** Grant the **minimum** access needed. Prefer **groups** over individual users.

### 1b — Grant chain (the three locks)

To `SELECT` from a view, a principal typically needs **all three**:

```
┌─────────────────────────────────┐
│ 1. USAGE on CATALOG  (main)     │ ← often implicit for workspace users
│ 2. USAGE on SCHEMA   (day04_…) │ ← required before seeing objects
│ 3. SELECT on VIEW    (flights…) │ ← the actual data access
└─────────────────────────────────┘
```

**Troubleshooting:** If analysts get *"permission denied"*, check **all three** — not just `SELECT`.

### Privilege reference

| Privilege | On | Meaning |
|-----------|-----|---------|
| `USAGE` | Catalog / schema | Can **see** the container and its children |
| `SELECT` | Table / view | Can **read** data |
| `MODIFY` | Table | Can `INSERT`, `UPDATE`, `DELETE` |
| `CREATE TABLE` | Schema | Can create new tables/views |
| `READ FILES` | External location | Can read files in that cloud path |
| `WRITE FILES` | External location | Can write files in that cloud path |

### 1c — Securable object hierarchy

```
metastore
 └── catalog
      └── schema
           ├── table
           ├── view
           ├── volume
           ├── function
           └── model
```

Each level is a **securable** — you can `GRANT` and `REVOKE` on it.

## 2 — Current identity (who am I?)

In [ ]:
# Useful when debugging "why can't I access this?"
try:
    spark.sql("SELECT current_user() AS me, current_catalog() AS cat, current_schema() AS sch").show(truncate=False)
except Exception:
    try:
        spark.sql("SELECT current_user() AS me").show()
    except Exception as e:
        print("current_user:", type(e).__name__, e)

## 3 — `SHOW GRANTS` (inspect who has access)

Output depends on your **role** (owner vs non-owner) and **DBR version**. Empty results are normal for non-owners.

In [ ]:
# 3a — Grants on Silver view
try:
    spark.sql(f"SHOW GRANTS ON VIEW {SILVER_V}").show(truncate=False)
except Exception as e:
    print("SHOW GRANTS ON VIEW:", type(e).__name__, str(e)[:120])

In [ ]:
# 3b — Grants on schema
try:
    spark.sql(f"SHOW GRANTS ON SCHEMA {FULL_SCHEMA}").show(truncate=False)
except Exception as e:
    print("SHOW GRANTS ON SCHEMA:", type(e).__name__, str(e)[:120])

In [ ]:
# 3c — Grants on multiple views (compare)
for v in [GOLD_V, DEST_V, MASKED_V, TAGGED_V]:
    print(f"=== GRANTS ON {v.split('.')[-1]} ===")
    try:
        spark.sql(f"SHOW GRANTS ON VIEW {v}").show(truncate=False)
    except Exception as e:
        print(f"  {type(e).__name__}: {str(e)[:80]}")

## 4 — `GRANT` / `REVOKE` templates (detailed)

Replace `<group>` with an Entra ID group name. In production, someone with ownership on the schema (or equivalent admin rights) runs these statements.

### 4a — Full access for analysts (Silver + Gold views)

```sql
-- Step 1: schema access
GRANT USAGE ON SCHEMA main.day04_u01_lab TO `data-analysts`;

-- Step 2: view-level SELECT
GRANT SELECT ON VIEW main.day04_u01_lab.flights_silver_v TO `data-analysts`;
GRANT SELECT ON VIEW main.day04_u01_lab.flights_gold_v   TO `data-analysts`;
```

### 4b — Restricted access (projection view only)

```sql
-- Analysts see ONLY destination + count — no origin, no route key
GRANT SELECT ON VIEW main.day04_u01_lab.flights_silver_dest_only_v TO `restricted-analysts`;
-- Do NOT grant on flights_silver_v — they should not see all columns
```

### 4c — Masked access (sensitive column hidden)

```sql
GRANT SELECT ON VIEW main.day04_u01_lab.flights_silver_masked_v TO `external-partners`;
```

### 4d — Revoke

```sql
REVOKE SELECT ON VIEW main.day04_u01_lab.flights_silver_v FROM `data-analysts`;
```

### 4e — Catalog-level USAGE (if required by your org)

```sql
GRANT USAGE ON CATALOG main TO `data-analysts`;
```

### 4f — Grant design exercise (discussion)

| Consumer group | Views they should access | Reason |
|----------------|--------------------------|--------|
| **Full analysts** | Silver, Gold | Need route-level detail |
| **Dashboard team** | Gold only | Only aggregates |
| **External partner** | Masked view | Origin is confidential |
| **US-focused team** | US-only view | Row-filtered for relevance |
| **Data engineer** | All + underlying Delta paths | Pipeline debugging |

**Exercise:** On paper, write the **GRANT** SQL for each row above.

## 5 — `information_schema` (metadata queries — 4 examples)

Each catalog exposes an `information_schema` with views like `tables`, `columns`, `views`, `schemata`. Column names and access vary by DBR — all wrapped in `try/except`.

In [ ]:
# 5a — Tables in your schema
try:
    spark.sql(
        "SELECT table_catalog, table_schema, table_name, table_type "
        + f"FROM {CATALOG}.information_schema.tables "
        + f"WHERE LOWER(table_schema) = LOWER('{SCHEMA_NAME}') "
        + "ORDER BY table_name"
    ).show(truncate=False)
except Exception as e:
    print("information_schema.tables:", type(e).__name__, str(e)[:120])

In [ ]:
# 5b — Columns of a specific view
try:
    spark.sql(
        "SELECT column_name, data_type, is_nullable "
        + f"FROM {CATALOG}.information_schema.columns "
        + f"WHERE table_schema = '{SCHEMA_NAME}' AND table_name = 'flights_silver_v' "
        + "ORDER BY ordinal_position"
    ).show(50, truncate=False)
except Exception as e:
    print("information_schema.columns:", type(e).__name__, str(e)[:120])

In [ ]:
# 5c — View definitions
try:
    spark.sql(
        "SELECT table_name, view_definition "
        + f"FROM {CATALOG}.information_schema.views "
        + f"WHERE table_schema = '{SCHEMA_NAME}'"
    ).show(truncate=False)
except Exception as e:
    print("information_schema.views:", type(e).__name__, str(e)[:120])

In [ ]:
# 5d — Schema metadata
try:
    spark.sql(
        "SELECT catalog_name, schema_name, schema_owner, comment "
        + f"FROM {CATALOG}.information_schema.schemata "
        + f"WHERE schema_name = '{SCHEMA_NAME}'"
    ).show(truncate=False)
except Exception as e:
    print("information_schema.schemata:", type(e).__name__, str(e)[:120])

In [ ]:
# 5e — DESCRIBE SCHEMA EXTENDED
try:
    spark.sql(f"DESCRIBE SCHEMA EXTENDED {FULL_SCHEMA}").show(truncate=False)
except Exception as e:
    print("DESCRIBE SCHEMA EXTENDED:", type(e).__name__, str(e)[:120])

## 6 — System lineage metadata (optional — M-7 Lab 20)

Some workspaces expose `system.information_schema.table_lineage` / `column_lineage`. Often **restricted** — treat failures as expected.

> Fallback: use **Catalog Explorer → Lineage** tab (Section 7).

In [ ]:
for name, q in [
    ("table_lineage",  "SELECT * FROM system.information_schema.table_lineage LIMIT 20"),
    ("column_lineage", "SELECT * FROM system.information_schema.column_lineage LIMIT 20"),
]:
    print(f"=== {name} ===")
    try:
        spark.sql(q).show(truncate=False)
    except Exception as e:
        print(f"  {type(e).__name__}: {str(e)[:100]}")

In [ ]:
# Alternative: system.lineage.table_lineage (Tata Lab 16 pattern)
try:
    spark.sql("SELECT * FROM system.lineage.table_lineage LIMIT 20").show(truncate=False)
except Exception as e:
    print("system.lineage.table_lineage:", type(e).__name__, str(e)[:100])

## 7 — Data lineage in Catalog Explorer (UI walkthrough)

### Step-by-step

1. In the left rail → **Data** (or **Catalog**)
2. Navigate to **`main` → `day04_uXX_lab`**
3. Click **`flights_silver_v`**
4. Open the **Lineage** tab

### What you should see

- **Upstream:** The Delta path under `data/day03/silver/flights_clean`
- **Downstream:** Any queries or views that read from this view

### What you might NOT see

- Views over `delta.\`path\`` sometimes show **limited** lineage until jobs write through governed paths
- Column-level lineage depends on runtime support

### Exercise

1. Open lineage for **`flights_silver_v`** (full projection)
2. Open lineage for **`flights_silver_dest_only_v`** (projection) — compare column coverage
3. Open lineage for **`flights_gold_v`** — note the different upstream Delta path
4. Note any differences and discuss with your partner

## 8 — Audit logging (concept — M-7 Lab 20 / Tata Lab 16)

### 8a — Where audit data lives

| Source | How to access | Who needs it |
|--------|---------------|--------------|
| **Azure Diagnostic Settings** | Azure Portal → Databricks resource → Diagnostic settings → Log Analytics / Storage | Security / compliance team |
| **`system.access.audit`** | SQL (if enabled) | Platform admin |
| **Workspace audit logs** | Account console | Account admin |

### 8b — What audit logs capture

- **Who** accessed what (user email, SP ID)
- **When** (timestamp)
- **What action** (SELECT, INSERT, CREATE VIEW, GRANT, etc.)
- **From where** (notebook, job, SQL warehouse)

### 8c — Azure Diagnostic setup (steps)

1. Open **Azure Portal** → your Databricks resource
2. **Diagnostic Settings** → **Add diagnostic setting**
3. Enable **Databricks Audit Logs**
4. Send to **Log Analytics** or **Storage Account**
5. Query with **KQL** in Log Analytics or **SQL** in system tables

In [ ]:
# 8d — system.access.audit (often restricted — try/except)
try:
    spark.sql("SELECT * FROM system.access.audit ORDER BY event_time DESC LIMIT 15").show(truncate=False)
except Exception as e:
    print("system.access.audit:", type(e).__name__, str(e)[:140])
    print("→ Use Azure Diagnostic Logs or Account Console instead")

In [ ]:
# 8e — Tata Lab 16 pattern: audit query on a specific table
try:
    spark.sql(f"""
SELECT user_identity.email, action_name, request_params.full_name_arg, event_time
FROM system.access.audit
WHERE request_params.full_name_arg LIKE '%day04%'
ORDER BY event_time DESC LIMIT 20
""").show(truncate=False)
except Exception as e:
    print("Filtered audit query:", type(e).__name__, str(e)[:100])

## 9 — Row-Level Security & Column Masking (theory + demo)

### 9a — Theory: three approaches to data protection in views

| Approach | How | Our lab example |
|----------|-----|-----------------|
| **Row filter (view WHERE)** | `CREATE VIEW ... WHERE country = 'US'` | `flights_silver_us_only_v` |
| **Column projection (view SELECT)** | `CREATE VIEW ... SELECT col1, col2` | `flights_silver_dest_only_v` |
| **Column masking (view expression)** | `SELECT '***' AS col` | `flights_silver_masked_v` |

### Advanced (UC Premium features)

| Feature | SQL | Scope |
|---------|-----|-------|
| **Row access policy** | `CREATE ROW ACCESS POLICY ...` | Applied to a table; evaluated per-user |
| **Column mask** | `ALTER TABLE ... ALTER COLUMN ... SET MASKING POLICY ...` | Hides PII per-user |

These depend on Databricks runtime and SKU; this course only introduces the concepts.

### 9b — Masked view demo (Tata Lab 16 style)

We already created `flights_silver_masked_v` in notebook **01**. Let's verify it works and compare with the unmasked view.

In [ ]:
# Unmasked: see real origin
print("=== UNMASKED (full Silver) ===")
spark.sql(f"SELECT DEST_COUNTRY_NAME, ORIGIN_COUNTRY_NAME, count FROM {SILVER_V} LIMIT 5").show(truncate=False)

# Masked: origin is '***'
print("=== MASKED ===")
spark.sql(f"SELECT DEST_COUNTRY_NAME, ORIGIN_COUNTRY_NAME, count FROM {MASKED_V} LIMIT 5").show(truncate=False)

In [ ]:
# Can you reverse the mask? Only if you have access to the full view or the Delta path.
# The masked view is safe to GRANT to external partners.
print("Masked view columns:", spark.table(MASKED_V).columns)
print("Full Silver columns:", spark.table(SILVER_V).columns)

### 9c — Production masking pattern (SQL template)

```sql
-- Create a masking function (UC Premium)
-- CREATE FUNCTION mask_origin(val STRING)
--   RETURNS STRING
--   RETURN CASE WHEN is_member('data-admins') THEN val ELSE '***' END;

-- Apply to a table column
-- ALTER TABLE main.prod.flights ALTER COLUMN ORIGIN_COUNTRY_NAME SET MASKING POLICY mask_origin;
```

**Key insight:** UC masking is **per-user** — admins see real values, analysts see masked. View-based masking (our approach) is **per-view** — everyone who reads the masked view sees `***`.

## 10 — Service principals & Jobs (concept)

| Principal type | Used for | GRANT syntax |
|---------------|----------|--------------|
| **User** (email) | Interactive notebooks | `GRANT ... TO \`user@contoso.com\`` |
| **Group** (Entra ID) | Team access | `GRANT ... TO \`data-analysts\`` |
| **Service principal** | Jobs, automation | `GRANT ... TO \`svc-etl-prod\`` |

A **Job** running as an SP needs the **same grant chain** (USAGE + SELECT) as a human user.

```sql
-- Example (replace principal name as appropriate):
-- GRANT USAGE ON SCHEMA main.day04_u01_lab TO `svc-databricks-etl`;
-- GRANT SELECT ON VIEW main.day04_u01_lab.flights_gold_v TO `svc-databricks-etl`;
```

## 11 — Additional metadata exploration

In [ ]:
# 11a — information_schema: all schemas matching 'day04%'
try:
    spark.sql(
        "SELECT schema_name, schema_owner, comment "
        + f"FROM {CATALOG}.information_schema.schemata "
        + "WHERE LOWER(schema_name) LIKE 'day04%' ORDER BY schema_name"
    ).show(30, truncate=False)
except Exception as e:
    print(type(e).__name__, str(e)[:120])

In [ ]:
# 11b — information_schema: columns of Gold view
try:
    spark.sql(
        "SELECT column_name, data_type "
        + f"FROM {CATALOG}.information_schema.columns "
        + f"WHERE table_schema = '{SCHEMA_NAME}' AND table_name = 'flights_gold_v' "
        + "ORDER BY ordinal_position"
    ).show(30, truncate=False)
except Exception as e:
    print(type(e).__name__, str(e)[:120])

In [ ]:
# 11c — system catalogs (often restricted)
try:
    spark.sql("SELECT * FROM system.information_schema.catalogs LIMIT 20").show(truncate=False)
except Exception as e:
    print("system catalogs:", type(e).__name__, str(e)[:120])

In [ ]:
# 11d — Column comparison: full vs projection vs masked
for v in ["flights_silver_v", "flights_silver_dest_only_v", "flights_silver_masked_v"]:
    cols = spark.table(f"{FULL_SCHEMA}.{v}").columns
    print(f"  {v:40s} → {len(cols)} cols: {cols}")

In [ ]:
# 11e — EXPLAIN COST comparison (if supported)
for label, view in [("full_silver", SILVER_V), ("dest_only", DEST_V)]:
    try:
        print(f"=== EXPLAIN COST {label} ===")
        spark.sql(f"EXPLAIN COST SELECT * FROM {view} LIMIT 1000").show(truncate=False)
    except Exception as e:
        print(f"  EXPLAIN COST: {type(e).__name__}")

## 12 — Compliance checklist (discussion)

Review with your partner — check off what you've verified in this lab:

- [ ] **Ownership** of `day04_uXX_lab` documented (DESCRIBE SCHEMA)
- [ ] **Grants** use **groups**, not individual users
- [ ] **Projection views** exist for low-sensitivity consumers
- [ ] **Masked views** exist for PII-sensitive consumers
- [ ] **Row-filtered views** exist for geo-restricted teams
- [ ] **Audit** logs are retained per policy (Azure Diagnostic or system tables)
- [ ] **Lineage** is reviewed before schema changes on Silver/Gold paths
- [ ] **Comments** added to views for discoverability in Catalog Explorer

## 13 — Practice drills (type answers in comments or run cells)

In [ ]:
# 13a — List all views in your schema
try:
    spark.sql(f"SHOW VIEWS IN SCHEMA {FULL_SCHEMA}").show(truncate=False)
except Exception:
    spark.sql(f"SHOW TABLES IN {FULL_SCHEMA}").show(truncate=False)

In [ ]:
# 13b — Count views
n = spark.sql(f"SHOW TABLES IN {FULL_SCHEMA}").count()
print(f"Objects in {FULL_SCHEMA}: {n}")

In [ ]:
# 13c — Grants on schema (count rows)
try:
    g = spark.sql(f"SHOW GRANTS ON SCHEMA {FULL_SCHEMA}").count()
    print(f"Grant rows on schema: {g}")
except Exception:
    print("SHOW GRANTS not available for non-owner")

In [ ]:
# 13d — Gold summary statistics
spark.table(GOLD_V).describe().show()

In [ ]:
# 13e — Silver summary statistics
spark.table(SILVER_V).describe().show()

In [ ]:
# 13f — Verify tagged view has extra columns
tagged_cols = set(spark.table(TAGGED_V).columns) - set(spark.table(SILVER_V).columns)
print("Extra columns in tagged view:", tagged_cols)

In [ ]:
# 13g — Verify US-only view filtering
us_dests = spark.table(US_V).select("DEST_COUNTRY_NAME").distinct().collect()
print("Distinct destinations in US-only view:", [r[0] for r in us_dests])

## 14 — Challenges

Short written answers or SQL where noted. Discuss in class if you want to validate your reasoning.

### C1 — Write the **three-step grant chain** (USAGE + USAGE + SELECT) for a new analyst group called `finance-readers` on `flights_gold_v`.

In [ ]:
# Your answer:


### C2 — Why is `REVOKE ... CASCADE` dangerous in shared schemas? (1–2 sentences)

In [ ]:
# Your answer:


### C3 — Name **two** places to see lineage besides SQL (`system.*`).

In [ ]:
# Your answer:


### C4 — Propose a grant plan: which group gets `SELECT` on **Gold only** vs **Silver + Gold** vs **masked only**.

In [ ]:
# Your answer:


### C5 — If `system.access.audit` fails, what is your **fallback** for audit evidence?

In [ ]:
# Your answer:


### C6 — Explain why `DROP VIEW` does **not** delete the underlying Delta files.

In [ ]:
# Your answer:


## 15 — Governance reading & reflection (offline OK)

Discuss with a partner or journal **2 bullets** you will apply per topic.

### Data stewardship vs data engineering

Stewards define meaning, retention, access classes. Engineers implement pipelines. UC **COMMENT**, **tags**, and **grants** bridge both.

### PII and views

Even with `SELECT` limited to a view, joins and inference can leak patterns. Combine projection views, masks, and training.

### Breaking changes to Silver

If Day 3 renames a column, views may fail. Mitigation: version contracts, compatibility tests, lineage review.

### Cost of wide `SELECT *`

Prefer narrow views for BI extracts — reduces scan cost. Ties to FinOps and sustainability.

### Multi-workspace governance

Same metastore serves dev and prod workspaces with different catalog access. Clarify where `GRANT` is evaluated.

### Dynamic views vs static views

Our views point at Delta paths. If underlying data changes, next `SELECT` sees new data — `GRANT SELECT` does not freeze data.

### Delta Sharing

Protocol to share data across organizations with recipient grants — beyond single-workspace UC. Not configured in this lab.

### Incident response playbook

1) Revoke broad grants  2) Identify objects via lineage  3) Rotate credentials  4) Preserve audit logs

## 16 — Quiz (no code — answer in your notes)

1. What privilege is **almost always** required before `SELECT` on a view?
2. Why use **views** instead of handing analysts raw `abfss://` paths?
3. Who typically creates **external locations**?
4. Can two **different** views reference the **same** Delta path?
5. Does `DROP VIEW` delete Parquet files?
6. What is the difference between **view-based masking** and **UC column masking**?
7. Name **three** things captured in audit logs.
8. Why should grants target **groups** instead of **individual users**?

Review these questions during the session or as take-home reflection.

## Summary

- Namespace: `catalog.schema.object`.
- This lab used several view patterns over shared `delta.` ABFS paths.
- Security: typically `USAGE` on catalog/schema and `SELECT` on the object; prefer groups for grants.
- Narrow or masked views limit exposure; row filters in views are a simple pattern; UC policies are the advanced case.
- Discovery: `SHOW`, `DESCRIBE`, `information_schema` where you have access.
- Lineage: Catalog Explorer; optional system metadata views depending on workspace.

Next: Day 5 — Delta Lake fundamentals.